In [1]:
# ------------------------------------
# 載入套件與環境設定
# ------------------------------------

import os
import sys

# 取得目前工作目錄，並加入 sys.path，確保能 import 同資料夾的 train_save 模組
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0,current_dir)

# 匯入訓練函式：訓練模型並序列化存成 salary_model.joblib
from train_save import train_and_save_model

print("環境準備完畢 已成功載入 trantrain_and_save_model 模組ˋ")


環境準備完畢 已成功載入 trantrain_and_save_model 模組ˋ


In [2]:
# ------------------------------------
# 定義 Pydantic 訓練模型(與app.py相同)
# 用途：透過 Pydantic 定義 API 的請求/回應「格式」與欄位驗證規則
# ------------------------------------
from pydantic import BaseModel,Field
from pprint import pprint   # 美化輸出 dict，方便查看結構

# 訓練請求的參數模型（POST /train 時前端需傳入這些欄位）
class TrainConfig(BaseModel):
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1 , le=0.5)   # 測試集比例 0.1~0.5
    random_state: int = Field(76, description="隨機種子", ge=0)                     # 隨機種子，固定結果可重現
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)")
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge= 0.001, le=100.0)  # 正則化強度

# 訓練完成後的回應模型（包含各項評估指標）
class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")            # 例如 "success"
    r2: float = Field(..., description="測試集 R-squared 決定係數")  # 模型解釋力
    coef: list[float] = Field(..., description="特徵權重係數列表")   # 每個特徵的斜率
    intercept: float = Field(..., description="截距")               # 迴歸線截距
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射")  # 特徵名稱 -> 係數
    model_type: str = Field(..., description="模型演算法類型")
    alpha: float = Field(..., description="正則化強度 alpha")
    train_time: float = Field(..., description="訓練耗時 (秒)")
    message:str = Field(..., description="提示訊息")

# 呼叫 model_json_schema() 把 Pydantic 模型轉成 JSON Schema 並輸出，確認欄位定義
print("TranConfig(BaseModel)")
pprint(TrainConfig.model_json_schema())
print("==============================")
print("TrainResult(BaseModel)")
pprint(TrainResult.model_json_schema())

TranConfig(BaseModel)
{'properties': {'alpha': {'default': 1.0,
                          'description': '正則化強度 alpha (適用於 Lasso 與 Ridge)',
                          'maximum': 100.0,
                          'minimum': 0.001,
                          'title': 'Alpha',
                          'type': 'number'},
                'model_type': {'default': 'LinearRegression',
                               'description': '模型演算法類型 (LinearRegression, '
                                              'Lasso, Ridge)',
                               'title': 'Model Type',
                               'type': 'string'},
                'random_state': {'default': 76,
                                 'description': '隨機種子',
                                 'minimum': 0,
                                 'title': 'Random State',
                                 'type': 'integer'},
                'test_size': {'default': 0.2,
                              'description': '測試集分割比例',
              

In [3]:
# ------------------------------------
# 拆解 train_and_save_model() 底層訓練與序列化
# 直接呼叫訓練函式，用 Ridge 演算法 + alpha=10 訓練，並存成 joblib
# ------------------------------------
from train_save import train_and_save_model
res_ridge:dict = train_and_save_model(
    test_size=0.2,        # 測試集比例 20%
    random_state=76,      # 隨機種子
    model_type="Ridge",  # 使用嶺迴歸（L2 正則化）
    alpha=10.0            # 正則化強度較高 → 係數會被壓得更小
)
pprint(res_ridge)   # 印出回傳的訓練結果（r2、係數、耗時等）

開始訓練 Ridge 嶺迴歸(α=10.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\__2026_07_03__\backend\0807\salary_model.joblib...
模型儲存成功！
{'alpha': 10.0,
 'coef': [3.915606705818322,
          10.029103401270465,
          -1.4644383465780102,
          -1.182860911975303,
          2.1482340576072554],
 'feature_coefs': {'City_城市A': -1.4644383465780102,
                   'City_城市B': -1.182860911975303,
                   'City_城市C': 2.1482340576072554,
                   'EducationLevel': 10.029103401270465,
                   'YearsExperience': 3.915606705818322},
 'intercept': 51.228571428571435,
 'message': 'Ridge 嶺迴歸(α=10.0) 模型訓練完成並儲存成功！',
 'model_type': 'Ridge',
 'r2': 0.8253872705107945,
 'status': 'success',
 'train_time': 0.025552034378051758}


# 理解 load_model_state() 全域動態更新機制

In [4]:
import joblib

# 指定模型檔案路徑，並建立空的 MODEL_STATE 全域字典
current_dir = os.getcwd()
model_path = os.path.join(current_dir,"salary_model.joblib")
MODEL_STATE = {}

# 全域動態更新機制：重新訓練後呼叫此函式，即可讓預測用到最新模型
def load_model_state():
    global MODEL_STATE
    # 如果沒有 joblib 檔案，先執行訓練產生檔案
    if not os.path.exists(model_path):
        train_and_save_model() #如果沒有joblib要先執行上一個模組產生joblib
    # 從檔案載入整個字典（含模型、預處理器、元數據）
    model_data = joblib.load(model_path)
    # 有值要全部清掉，確保每次載入的都是全新的狀態，不會殘留舊模型
    MODEL_STATE.clear() #有值要全部清掉 要一個全新的
    # 把各物件放進全域字典，供後續 /predict 使用
    MODEL_STATE.update({
        "model": model_data["model"],      # 迴歸模型
        "oe" : model_data["oe"],           # 學歷編碼器
        "ohe" :model_data["ohe"],          # 城市 OneHot 編碼器
        "scaler" :model_data["scaler"],    # 標準化器
        "r2" :model_data["r2"],            # 模型 R² 分數
        "feature_names" :model_data["feature_names"],   # 特徵名稱
        "feature_coefs" :model_data["feature_coefs"],   # 特徵權重映射
        "model_type": model_data["model_type"],         # 演算法類型
        "alpha": model_data["alpha"],                   # 正則化強度
    })
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")
load_model_state()   # 啟動時呼叫一次，初始化全域狀態

✅ MODEL_STATE 已成功更新！當前模型：Ridge，R² Score：0.8254


In [6]:
# ------------------------------------
# 定義預測的class
# ------------------------------------
# 預測請求的資料模型（POST /predict 前端需傳入）
class SalaryInput(BaseModel):
    years_experience: float = Field(..., ge=0.0, le=50.0)   # 工作年資，限制 0~50
    education_level:str                                    # 學歷：高中以下 / 大學 / 碩士以上
    city: str                                              # 城市：城市A / 城市B / 城市C

# 預測結果的回應資料模型
class SalaryOutput(BaseModel):
    predicted_salary: float        # 預測月薪
    estimated_annual_salary: float # 預估年薪（月薪 × 14）
    

In [7]:
# ------------------------------------
# 預測api
# 把「編碼 → 組特徵 → 標準化 → 預測」流程包成函式，模擬 /predict 端點
# ------------------------------------
import pandas as pd
import numpy as np

def predict_api(years_experience:float, education_level:str, city:str) -> dict:
    # 從全域狀態取出預處理器與模型
    oe = MODEL_STATE["oe"]       # 學歷編碼器
    ohe = MODEL_STATE["ohe"]     # 城市 OneHot 編碼器
    scaler = MODEL_STATE["scaler"]  # 標準化器
    model = MODEL_STATE["model"]    # 迴歸模型

    # 1. 學歷文字轉數值（高中以下=0、大學=1、碩士以上=2）
    edu_encoded = int(oe.transform(pd.DataFrame([[education_level]], columns=["EducationLevel"]))[0][0])
    # 2. 城市文字轉 OneHot 向量（例：城市C -> [0,0,1]）
    city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
    city_cols = ohe.get_feature_names_out(['City'])
    # 3. 組合特徵列，欄位順序要與訓練時一致
    feature_row = [years_experience, edu_encoded] + list(city_vector[0])
    features = pd.DataFrame([feature_row],columns=["YearsExperience", "EducationLevel"] + list(city_cols))
    # 4. 標準化後送入模型預測月薪
    X_scaled = scaler.transform(features)
    predicted_salary = float(model.predict(X_scaled)[0])
    # 5. 回傳月薪與年薪（×14 個月）
    return {
        "predicted_salary": predicted_salary,
        "estimated_annual_salary": predicted_salary * 14
    }


# 測試：10 年年資、大學學歷、城市C
predict_api(years_experience=10.0, education_level="大學", city="城市C")

{'predicted_salary': 61.07802879415863,
 'estimated_annual_salary': 855.0924031182208}

In [5]:
# ------------------------------------
# 流程串成 train_api 函式
# 把「接收 TrainConfig 參數 → 呼叫訓練函式 → 錯誤處理」包成可重用的函式
# ------------------------------------
from fastapi import HTTPException
def train_api(config: TrainConfig) -> dict :
    try:
        # 把 Pydantic 驗證過後的參數丟進訓練函式
        res = train_and_save_model(
        test_size= config.test_size,
        random_state= config.random_state,
        model_type= config.model_type,
        alpha= config.alpha     
        )
    except Exception as e:
        # 任一環節出錯就拋出 HTTP 500，讓 API 呼叫端知道失敗
        raise HTTPException(status_code=500,detail="線上訓練失敗")
    return res


In [8]:
# ------------------------------------
# 使用 FastAPI 建立一個 API，讓外部程式可以透過 HTTP POST 呼叫模型訓練功能
# 模擬使用者整合測試節點是否正常運作
# ------------------------------------
from fastapi import FastAPI
from fastapi.testclient import TestClient   # 不需真的開啟伺服器即可模擬 HTTP 請求

# 建立迷你版 FastAPI 應用（與 app.py 結構一致）
mini_api = FastAPI()

# POST /train：接收 TrainConfig，呼叫 train_api 重新訓練
@mini_api.post("/train", response_model=TrainResult)
def train_endpoint(config:TrainConfig):
    res = train_api(config=config)
    return res

# POST /predict：接收 SalaryInput，用目前 MODEL_STATE 的模型做預測
@mini_api.post("/predict", response_model=SalaryOutput)
def predict_endpoint(payload:SalaryInput):
    # 取出預處理器與模型
    oe = MODEL_STATE["oe"]
    ohe = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model = MODEL_STATE["model"]

    # 學歷文字 -> 數值
    edu_encoded = int(oe.transform(pd.DataFrame([[payload.education_level]], columns=["EducationLevel"]))[0][0])
    # 城市文字 -> OneHot 向量
    city_vector = ohe.transform(pd.DataFrame([[payload.city]], columns=["City"]))
    city_cols = ohe.get_feature_names_out(['City'])
    # 組合特徵列並標準化
    feature_row = [payload.years_experience, edu_encoded] + list(city_vector[0])
    features = pd.DataFrame([feature_row],columns=["YearsExperience", "EducationLevel"] + list(city_cols))
    X_scaled = scaler.transform(features)
    predicted_salary = float(model.predict(X_scaled)[0])
    # 回傳月薪與年薪（×14）
    return SalaryOutput(
        predicted_salary=predicted_salary,
        estimated_annual_salary= predicted_salary * 14
    )
    
    

# 使用 TestClient 模擬發送 POST /train 請求（Lasso + alpha=5）
client = TestClient(mini_api)
response = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "Lasso",
    "alpha": 5.0
})
print("【重訓 Lasso 結果】")
print("HTTP 狀態碼:", response.status_code)
pprint(response.json())

# 再模擬發送 POST /predict 請求（5.3 年年資、碩士以上、城市A）
response1 = client.post("/predict",json={
    "years_experience": 5.3,
    "education_level": "碩士以上",
    "city": "城市A"
})

print("\n預測月薪:", response1.json()["predicted_salary"])

開始訓練 Lasso 迴歸(α=5.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\__2026_07_03__\backend\0807\salary_model.joblib...
模型儲存成功！
【重訓 Lasso 結果】
HTTP 狀態碼: 200
{'alpha': 5.0,
 'coef': [0.21476373685969363, 11.506443086096304, -0.0, -0.0, 0.0],
 'feature_coefs': {'City_城市A': -0.0,
                   'City_城市B': -0.0,
                   'City_城市C': 0.0,
                   'EducationLevel': 11.506443086096304,
                   'YearsExperience': 0.21476373685969363},
 'intercept': 51.228571428571435,
 'message': 'Lasso 迴歸(α=5.0) 模型訓練完成並儲存成功！',
 'model_type': 'Lasso',
 'r2': 0.8441175875255694,
 'status': 'success',
 'train_time': 0.006000041961669922}

預測月薪: 58.820855109818105


C:\Users\User\Documents\GitHub\__2026_07_03__\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
